# Actor-Critic

Теорема о градиенте стратегии связывает градиент целевой функции  и градиент самой стратегии:

$$\nabla_\theta J(\theta) = \mathbb{E}_\pi [Q^\pi(s, a) \nabla_\theta \ln \pi_\theta(a \vert s)]$$

Встает вопрос, как оценить $Q^\pi(s, a)$? В чистом policy-based алгоритме REINFORCE используется отдача $G_t$, полученная методом Монте-Карло в качестве несмещенной оценки $Q^\pi(s, a)$. В Actor-Critic же предлагается отдельно обучать нейронную сеть Q-функции — критика.

Актор-критиком часто называют обобщенный фреймворк (подход), нежели какой-то конкретный алгоритм. Как подход актор-критик не указывает, каким конкретно [policy gradient] методом обучается актор и каким [value based] методом обучается критик. Таким образом актор-критик задает целое [семейство](https://proceedings.neurips.cc/paper_files/paper/1999/file/6449f44a102fde848669bdd9eb6b76fa-Paper.pdf) различных алгоритмов. Рекомендую в качестве шпаргалки использовать упомянутый в тетрадке с REINFORCE [пост из блога Lilian Weng](https://lilianweng.github.io/posts/2018-04-08-policy-gradient/), посвященный наиболее популярным алгоритмам семейства актор-критиков

В данной тетрадке познакомимся с наиболее простым вариантом актор-критика, который так и называют Actor-Critic:

In [1]:
# Cтавим нужные зависимости, если это колаб
try:
    import google.colab
    COLAB = True
except ModuleNotFoundError:
    COLAB = False
    pass

if COLAB:
    !pip -q install "gymnasium[classic-control, atari, accept-rom-license]"
    !pip -q install piglet
    !pip -q install imageio_ffmpeg
    !pip -q install moviepy==1.0.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.5/67.5 kB 2.6 MB/s eta 0:00:00


In [2]:
from collections import deque

import gymnasium as gym
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.distributions import Categorical

%matplotlib inline

In [3]:
env = gym.make("CartPole-v1")
env.reset()

print(f'{env.observation_space=}')
print(f'{env.action_space=}')

n_actions = env.action_space.n
state_dim = env.observation_space.shape
print(f'Action_space: {n_actions} | State_space: {env.observation_space.shape}')

env.observation_space=Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
env.action_space=Discrete(2)
Action_space: 2 | State_space: (4,)


(1 балл)

In [9]:
def to_tensor(x, dtype=np.float32):
    if isinstance(x, torch.Tensor):
        return x
    x = np.asarray(x, dtype=dtype)
    x = torch.from_numpy(x)
    return x

def symlog(x):
    """Compute symlog values for a vector `x`. It's an inverse operation for symexp."""
    return torch.sign(x) * torch.log(torch.abs(x) + 1)

def symexp(x):
    """Compute symexp values for a vector `x`. It's an inverse operation for symlog."""
    return torch.sign(x) * (torch.exp(torch.abs(x)) - 1.0)


class SymExpModule(nn.Module):
    def forward(self, x):
        return symexp(x)

def select_action_eps_greedy(Q, state, epsilon):
    """Выбирает действие epsilon-жадно."""
    if not isinstance(state, torch.Tensor):
        state = torch.tensor(state, dtype=torch.float32)
    Q_s = Q(state).detach().numpy()

    ####### Здесь ваш код ########
    if np.random.rand() < epsilon:
        action = np.random.randint(len(Q_s))
    else:
        action = np.argmax(Q_s)
    ##############################

    action = int(action)
    return action

def sample_batch(replay_buffer, n_samples):
    n_samples = min(n_samples, len(replay_buffer))
    indices = np.random.choice(len(replay_buffer), n_samples, replace=False)
    batch = [replay_buffer[i] for i in indices]
    states, actions, rewards, next_states, terminateds = zip(*batch)

    return np.array(states), np.array(actions), np.array(rewards), np.array(next_states), np.array(terminateds)

## Shared-body Actor-Critic

Актор и критик могут обучаться в разных режимах — актор только on-policy (шаг обучения на текущей собранной подтраектории), а критик on-policy или off-policy (шаг обучения на текущей подтраектории или на батче из replay buffer). Это с одной стороны привносит гибкость в обучение, с другой — усложняет его.

Если актор и критик оба обучаются on-policy, то имеет смысл объединить их сетки в одну и делать общий шаг обратного распространения ошибки. Однако, если они обучаются в разных режимах (и с разной частотой обновления), то велика вероятность, что их шаги обучения могут начать конфликтовать в случае общего тела — для такого варианта намного предпочтительнее разделить их на разные подсети (либо аккуратно настраивать гиперпарметры, чтобы стабилизировать обучение). В целом, рекомендуется использовать общий энкодер наблюдений, а далее как можно скорее разделять головы.

Сделаем реализацию актор-критика с общим телом и с on-policy вариантом обучения.

In [5]:
class ActorBatch:
    def __init__(self):
        self.logprobs = []
        self.q_values = []

    def append(self, log_prob, q_value):
        self.logprobs.append(log_prob)
        self.q_values.append(q_value)

    def clear(self):
        self.logprobs.clear()
        self.q_values.clear()

(3 балла)

In [6]:
class ActorCriticNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dims):
        super().__init__()

        ####### Здесь ваш код ########
        layers = []
        prev_dim = state_dim
        for h_dim in hidden_dims:
            layers.extend([nn.Linear(prev_dim, h_dim), nn.ReLU()])
            prev_dim = h_dim
        self.net = nn.Sequential(*layers)

        self.actor_head = nn.Sequential(nn.Linear(prev_dim, action_dim), nn.Softmax(dim=-1))
        self.critic_head = nn.Linear(prev_dim, action_dim)
        ##############################

    def forward(self, state):
        ####### Здесь ваш код ########
        features = self.net(state)
        probs = self.actor_head(features)
        dist = Categorical(probs)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        q_values = self.critic_head(features)
        Q_s_a = q_values[action]
        ##############################

        return action, log_prob, Q_s_a

    def evaluate(self, state):
        ####### Здесь ваш код ########
        features = self.net(state)
        q_values = self.critic_head(features)
        ##############################
        return q_values

(6 баллов)

In [7]:
class ActorCriticAgent:
    def __init__(self, state_dim, action_dim, hidden_dims, lr, gamma, critic_rb_size):
        self.lr = lr
        self.gamma = gamma

        ####### Здесь ваш код ########
        self.actor_critic = ActorCriticNet(state_dim, action_dim, hidden_dims)
        self.opt = torch.optim.Adam(self.actor_critic.parameters(), lr=lr)
        ##############################

        self.actor_batch = ActorBatch()
        self.critic_rb = deque(maxlen=critic_rb_size)

    def act(self, state):
        ####### Здесь ваш код ########
        state_tensor = to_tensor(state)
        action, log_prob, q_value = self.actor_critic(state_tensor)
        self.actor_batch.append(log_prob, q_value.detach())
        action = action.item()
        ##############################

        return action

    def append_to_replay_buffer(self, s, a, r, next_s, terminated):
        ####### Здесь ваш код ########
        self.critic_rb.append((s, a, r, next_s, terminated))
        ##############################

    def evaluate(self, state):
        return self.actor_critic.evaluate(state)

    def update(self, rollout_size, critic_batch_size, critic_updates_per_actor):
        if len(self.actor_batch.q_values) < rollout_size:
            return

        self.opt.zero_grad()
        loss = self.update_critic(critic_batch_size, critic_updates_per_actor)
        loss += self.update_actor()
        loss.backward()

        self.opt.step()
        self.actor_batch.clear()
        self.critic_rb.clear()

    def update_actor(self):
        Q_s_a = to_tensor(self.actor_batch.q_values)
        logprobs = torch.stack(self.actor_batch.logprobs)

        ####### Здесь ваш код ########
        loss = -(logprobs * Q_s_a).mean()
        ##############################
        return loss

    def update_critic(self, batch_size, n_updates=1):
        ####### Здесь ваш код ########
        total_loss = 0
        for _ in range(n_updates):
            states, actions, rewards, next_states, terminateds = sample_batch(self.critic_rb, batch_size)
            loss = self.compute_td_loss(states, actions, rewards, next_states, terminateds)
            total_loss += loss
        ##############################
        return total_loss

    def compute_td_loss(
        self, states, actions, rewards, next_states, terminateds, regularizer=0.1
    ):
        s = to_tensor(states)
        a = to_tensor(actions, int).long()
        r = to_tensor(rewards)
        s_next = to_tensor(next_states)
        term = to_tensor(terminateds, bool)

        ####### Здесь ваш код ########
        Q_s = self.actor_critic.evaluate(s)
        Q_s_a = Q_s[torch.arange(len(a)), a]

        with torch.no_grad():
            Q_sn = self.actor_critic.evaluate(s_next)
            V_sn = Q_sn.max(dim=1)[0]
            V_sn[term] = 0

        target = r + self.gamma * V_sn
        td_error = Q_s_a - target
        ##############################

        loss = torch.mean(td_error ** 2)
        loss += regularizer * Q_s_a.mean()
        return loss

In [10]:
def run_actor_critic(
        env_name="CartPole-v1",
        hidden_dims=(128, 128), lr=5e-4,
        total_max_steps=200_000,
        train_schedule=16, replay_buffer_size=50000, batch_size=64, critic_updates_per_actor=4,
        eval_schedule=1000, smooth_ret_window=10, success_ret=200.
):
    env = gym.make(env_name)
    episode_return_history = deque(maxlen=smooth_ret_window)

    agent = ActorCriticAgent(
        state_dim=env.observation_space.shape[0], action_dim=env.action_space.n, hidden_dims=hidden_dims,
        lr=lr, gamma=.995, critic_rb_size=replay_buffer_size
    )

    s, _ = env.reset()
    done, episode_return = False, 0.
    eval = False

    for global_step in range(1, total_max_steps+1):
        a = agent.act(s)
        s_next, r, terminated, truncated, _ = env.step(a)
        episode_return += r
        done = terminated or truncated

        agent.append_to_replay_buffer(s, a, r, s_next, terminated)
        agent.update(train_schedule, batch_size, critic_updates_per_actor)

        if global_step % eval_schedule == 0:
            eval = True

        s = s_next
        if done:
            if eval:
                episode_return_history.append(episode_return)
                avg_return = np.mean(episode_return_history)
                print(f'{global_step=} | {avg_return=:.3f}')
                if avg_return >= success_ret:
                    print('Решено!')
                    break

            s, _ = env.reset()
            done, episode_return = False, 0.
            eval = False

run_actor_critic(eval_schedule=2000, total_max_steps=100_000)

global_step=2005 | avg_return=8.000
global_step=4004 | avg_return=9.500
global_step=6004 | avg_return=9.667
global_step=8002 | avg_return=11.250
global_step=10002 | avg_return=11.000
global_step=12005 | avg_return=12.167
global_step=14029 | avg_return=17.286
global_step=16052 | avg_return=22.625
global_step=18022 | avg_return=24.556
global_step=20071 | avg_return=35.900
global_step=22010 | avg_return=38.800
global_step=24026 | avg_return=40.700
global_step=26009 | avg_return=51.200
global_step=28011 | avg_return=53.900
global_step=30004 | avg_return=55.300
global_step=32040 | avg_return=58.100
global_step=34004 | avg_return=57.200
global_step=36029 | avg_return=57.000
global_step=38114 | avg_return=64.500
global_step=40047 | avg_return=61.900
global_step=42007 | avg_return=60.600
global_step=44037 | avg_return=68.200
global_step=46022 | avg_return=63.200
global_step=48007 | avg_return=69.400
global_step=50036 | avg_return=76.900
global_step=52065 | avg_return=89.700
global_step=54083 |